## Preprocessing & Feature Engineering

Workflow:
1. Load Data
2. Remove duplicate rows
3. Split into train / test sets
4. Encode nominal features (proto, service, state) with OneHotEncoder
5. Transform skewed numerical features (log transform)
6. Save processed datasets

In [1]:
# 1. Load data

import pandas as pd

df_train0 = pd.read_parquet("../data/UNSW_NB15_training-set.parquet")

In [2]:
# 2. Remove duplicate rows

df_train1 = df_train0.drop_duplicates()

print(f"Removed {df_train0.shape[0]-df_train1.shape[0]} duplicate rows")
print(f"Remaining rows: {df_train1.shape[0]}")

Removed 78519 duplicate rows
Remaining rows: 96822


In [3]:
# 3. Split into train / test sets

from sklearn.model_selection import train_test_split

train_size = 0.67
random_state = 42

X = df_train1.drop(columns=['label', 'attack_cat'])
y_label = df_train1['label']
y_attack_cat = df_train1['attack_cat']

X_train, X_test, y_label_train, y_label_test, y_cat_train, y_cat_test = train_test_split(
    X, y_label, y_attack_cat,
    train_size = train_size,
    random_state = random_state,
    stratify = y_label        # To keep the same proportion of label (attack/normal) in train and test
)

print(f"Train: {X_train.shape[0]} rows")
print(f"Test: {X_test.shape[0]} rows")

Train: 64870 rows
Test: 31952 rows


In [4]:
# 4. Encode nominal features (proto, service, state) with OneHotEncoder

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
import numpy as np

nominal_cols = ['proto', 'service', 'state']
transf_type = np.int32

encoder = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore', sparse_output = False, dtype = transf_type), nominal_cols),
    remainder='passthrough'
)

X_train_encoded = pd.DataFrame(encoder.fit_transform(X_train), columns = encoder.get_feature_names_out())
X_test_encoded = pd.DataFrame(encoder.transform(X_test), columns = encoder.get_feature_names_out())

X_train_encoded.columns = [c.split('__')[-1] for c in X_train_encoded.columns]
X_test_encoded.columns = [c.split('__')[-1] for c in X_test_encoded.columns]

In [5]:
# Test to ensure that X_train and X have the same number of columns

assert X_train_encoded.shape[1] == X_test_encoded.shape[1], "Train/test hanno numero diverso di colonne!"
print(X_train_encoded.shape, X_test_encoded.shape)

(64870, 185) (31952, 185)


In [6]:
# 5. Transform skewed numerical features (log transform)

numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

skewness = X_train[numerical_cols].skew().sort_values(ascending = False)

skew_threshold = 1.0
skewed_cols = skewness[skewness > skew_threshold].index.tolist()

print(f"{len(skewed_cols)} features selected for log transform (skewness > {skew_threshold}):")
print(skewed_cols)

X_train_encoded[skewed_cols] = X_train_encoded[skewed_cols].apply(np.log1p)
X_test_encoded[skewed_cols] = X_test_encoded[skewed_cols].apply(np.log1p)

print("\nLog transform applied.")

27 features selected for log transform (skewness > 1.0):
['trans_depth', 'response_body_len', 'sbytes', 'sloss', 'spkts', 'dinpkt', 'dloss', 'djit', 'dbytes', 'dpkts', 'sjit', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'sinpkt', 'ct_dst_sport_ltm', 'sload', 'dur', 'is_ftp_login', 'ct_ftp_cmd', 'ct_src_dport_ltm', 'rate', 'synack', 'ackdat', 'tcprtt', 'dload', 'smean', 'dmean']

Log transform applied.


In [7]:
# 6. Save processed datasets

import os

os.makedirs("../data/processed", exist_ok = True)

X_train_encoded.to_parquet("../data/processed/X_train.parquet")
X_test_encoded.to_parquet("../data/processed/X_test.parquet")

pd.DataFrame(y_label_train).to_parquet("../data/processed/y_label_train.parquet")
pd.DataFrame(y_label_test).to_parquet("../data/processed/y_label_test.parquet")

pd.DataFrame(y_cat_train).to_parquet("../data/processed/y_cat_train.parquet")
pd.DataFrame(y_cat_test).to_parquet("../data/processed/y_cat_test.parquet")

print("Processed datasets saved to ../data/processed")

Processed datasets saved to ../data/processed
